In [ ]:
# # 📓 Notebook 10: End-to-End Production Enterprise Search System Design

# ## Project: 09-enterprise-search-pipeline

# ---

# # 🎯 Learning Goal

# By the end of this notebook, you will understand:

# * How all previous notebooks fit together into a complete production system.
# * How enterprise search pipelines are deployed in real organizations.
# * How ingestion, indexing, retrieval, APIs, and dashboards interact.
# * How to handle continuous document updates.
# * How to scale BM25, FAISS, and Cross-Encoder services.
# * How to deploy the system using Docker and Kubernetes.
# * How to answer enterprise search system design interview questions.

# This notebook builds the **Production Deployment and System Design Layer** of the Enterprise Search Pipeline.

# ---

# # 🏗️ Complete Project Journey

# ```text
# 09-enterprise-search-pipeline/
# │
# ├── 01-building-enterprise-corpus.ipynb              ✅
# ├── 02-acl-and-metadata-indexing.ipynb               ✅
# ├── 03-bm25-lexical-search-engine.ipynb              ✅
# ├── 04-semantic-vector-search-faiss.ipynb            ✅
# ├── 05-hybrid-search-and-rrf-fusion.ipynb            ✅
# ├── 06-cross-encoder-reranking.ipynb                 ✅
# ├── 07-access-control-and-secure-search.ipynb        ✅
# ├── 08-enterprise-search-api.ipynb                   ✅
# ├── 09-streamlit-enterprise-search-dashboard.ipynb   ✅
# └── 10-end-to-end-production-enterprise-search.ipynb ◄ You are here
# ```

# ---

# # 🧠 Section 1: What Have We Actually Built?

# We have built a simplified version of the architecture used by:

# * Microsoft Enterprise Search.
# * Elastic Enterprise Search.
# * Glean AI.
# * Atlassian Rovo.
# * Internal corporate RAG systems.
# * AI-powered knowledge assistants.

# Our pipeline supports:

# ✅ Multi-source document ingestion.

# ✅ ACL-aware metadata indexing.

# ✅ BM25 lexical retrieval.

# ✅ SBERT + FAISS semantic retrieval.

# ✅ Hybrid Search using RRF.

# ✅ Cross-Encoder re-ranking.

# ✅ FastAPI serving.

# ✅ Streamlit UI.

# ---

# # 🏭 Section 2: High-Level Production Architecture

# ```text
#                     ┌─────────────────────────────┐
#                     │     Enterprise Sources      │
#                     │─────────────────────────────│
#                     │ • Slack                     │
#                     │ • Confluence                │
#                     │ • Jira                      │
#                     │ • SharePoint                │
#                     │ • Internal Wikis            │
#                     └──────────────┬──────────────┘
#                                    │
#                                    ▼
#                    ┌──────────────────────────────┐
#                    │   Ingestion & ETL Pipeline   │
#                    └──────────────┬───────────────┘
#                                   │
#                                   ▼
#                  ┌────────────────────────────────┐
#                  │ Corpus + ACL + Metadata Builder│
#                  └──────────────┬─────────────────┘
#                                 │
#               ┌─────────────────┼─────────────────┐
#               ▼                                   ▼
#       BM25 Index Builder                Embedding Generator
#               │                                   │
#               ▼                                   ▼
#        BM25 Inverted Index               FAISS Vector Index
#               └─────────────────┬─────────────────┘
#                                 ▼
#                    Enterprise Search Backend
#                                 │
#                   ┌─────────────┼─────────────┐
#                   ▼                           ▼
#             FastAPI Service          Authentication
#                   │                           │
#                   └─────────────┬─────────────┘
#                                 ▼
#                      Streamlit Dashboard
#                                 │
#                                 ▼
#                          Enterprise Users
# ```

# ---

# # 🧠 Section 3: Offline vs Online Pipeline

# ## Offline Pipeline (Batch Indexing)

# Runs every few minutes or hours.

# ```text
# Documents
#     │
#     ▼
# Extract Text
#     │
#     ▼
# Clean + Normalize
#     │
#     ▼
# Generate Metadata
#     │
#     ▼
# Create Embeddings
#     │
#     ▼
# Build BM25 + FAISS Indexes
#     │
#     ▼
# Save Artifacts
# ```

# ---

# ## Online Pipeline (Real-Time Search)

# Runs for every user query.

# ```text
# User Query
#      │
#      ▼
# Authenticate User
#      │
#      ▼
# ACL Filtering
#      │
#      ▼
# BM25 + FAISS Retrieval
#      │
#      ▼
# RRF Fusion
#      │
#      ▼
# Cross-Encoder Re-ranking
#      │
#      ▼
# JSON Response
#      │
#      ▼
# Dashboard Display
# ```

# ---

# # 📚 Section 4: Project Folder Structure

# ```text
# enterprise-search-project/
# │
# ├── data/
# │   ├── raw/
# │   └── processed/
# │       └── enterprise_corpus_acl.csv
# │
# ├── artifacts/
# │   ├── bm25_model.pkl
# │   ├── enterprise_faiss.index
# │   ├── document_embeddings.pkl
# │   ├── hybrid_search_config.pkl
# │   ├── cross_encoder_config.pkl
# │   ├── security_config.pkl
# │   └── api_config.pkl
# │
# ├── api/
# │   └── app.py
# │
# ├── dashboard/
# │   └── streamlit_app.py
# │
# ├── notebooks/
# │   ├── 01-...ipynb
# │   └── ...
# │
# ├── requirements.txt
# ├── Dockerfile
# ├── docker-compose.yml
# └── README.md
# ```

# ---

# # 🔄 Section 5: Continuous Document Updates

# ## Problem

# Suppose someone edits a Confluence page.

# Do we rebuild the entire FAISS index?

# **No.**

# ---

# ## Production Solution

# Assign each document a stable hash.

# ## Cell 1

# ```python
# import hashlib

# def generate_document_hash(
#     text
# ):

#     return hashlib.sha256(
#         text.encode(
#             "utf-8"
#         )
#     ).hexdigest()
# ```

# ---

# ## Cell 2

# ```python
# enterprise_df["document_hash"] = (
#     enterprise_df[
#         "search_text"
#     ].apply(
#         generate_document_hash
#     )
# )

# enterprise_df[
#     [
#         "doc_id",
#         "document_hash"
#     ]
# ].head()
# ```

# ---

# ## Incremental Update Pipeline

# ```text
# Document Updated
#         │
#         ▼
# Compute New Hash
#         │
#         ▼
# Hash Changed?
#    ┌────┴────┐
#    │         │
#   No        Yes
#    │         │
#  Skip    Re-embed Chunk
#               │
#               ▼
#      Update FAISS Entry
#               │
#               ▼
#       Update BM25 Index
# ```

# ---

# # ⚡ Section 6: Micro-Batching Index Updates

# Instead of rebuilding immediately:

# ```text
# Slack Update
# Confluence Update
# Jira Update
#       │
#       ▼
# Webhook Queue
#       │
#       ▼
# Kafka / RabbitMQ
#       │
#       ▼
# Embedding Worker Pool
#       │
#       ▼
# Update BM25 + FAISS
# ```

# Advantages:

# * Lower compute cost.
# * Better throughput.
# * Easier scaling.

# ---

# # 🐳 Section 7: Docker Deployment

# ## Dockerfile

# ## Cell 3

# ```dockerfile
# FROM python:3.11-slim

# WORKDIR /app

# COPY requirements.txt .

# RUN pip install --no-cache-dir -r requirements.txt

# COPY . .

# EXPOSE 8000

# CMD [
#     "uvicorn",
#     "api.app:app",
#     "--host",
#     "0.0.0.0",
#     "--port",
#     "8000"
# ]
# ```

# ---

# # 🐳 Section 8: Docker Compose

# ## Cell 4

# ```yaml
# version: "3.9"

# services:

#   enterprise-api:

#     build: .

#     container_name: enterprise-search-api

#     ports:
#       - "8000:8000"

#     restart: always

#   streamlit:

#     build: .

#     command: >
#       streamlit run
#       dashboard/streamlit_app.py
#       --server.port=8501

#     ports:
#       - "8501:8501"

#     depends_on:
#       - enterprise-api
# ```

# ---

# # ☸️ Section 9: Kubernetes Deployment (Conceptual)

# ```text
#                     Internet
#                         │
#                         ▼
#                 Load Balancer
#                         │
#             ┌───────────┼───────────┐
#             ▼                       ▼
#       API Pod 1               API Pod 2
#             │                       │
#             └───────────┬───────────┘
#                         ▼
#                  Shared FAISS Store
#                         │
#                         ▼
#               Shared Enterprise Corpus
# ```

# Kubernetes provides:

# * Horizontal scaling.
# * Auto-restart.
# * Load balancing.
# * Rolling deployments.

# ---

# # 📡 Section 10: End-to-End Request Lifecycle

# ## Example Query

# ```text
# "Find PHX-245 vector database migration documents."
# ```

# ### Step 1 — User

# ```text
# User → Streamlit Dashboard
# ```

# ### Step 2 — API

# ```text
# POST /search
# ```

# Request:

# ```json
# {
#   "user_name": "alice",
#   "query": "PHX-245 vector database migration",
#   "top_k": 5
# }
# ```

# ### Step 3 — Authentication

# ```text
# alice
#   │
#   ▼
# Engineering Role
# ```

# ### Step 4 — ACL Filtering

# Only Engineering + Public documents remain.

# ### Step 5 — Hybrid Retrieval

# ```text
# BM25 → Top 20
# FAISS → Top 20
# ```

# ### Step 6 — RRF

# Merge candidate lists.

# ### Step 7 — Cross-Encoder

# Score:

# * (Query, Doc1)
# * (Query, Doc2)
# * ...

# ### Step 8 — API Response

# ```json
# {
#   "results": [
#     {
#       "doc_id": "DOC001",
#       "title": "Enterprise Search Design",
#       "score": 9.41
#     }
#   ]
# }
# ```

# ### Step 9 — Streamlit

# Display interactive dashboard.

# ---

# # 📊 Section 11: Complete Enterprise Search Flow

# ```text
#                  Enterprise Documents
#                            │
#                            ▼
#                Ingestion & ETL Pipeline
#                            │
#                            ▼
#              Metadata + ACL Enrichment
#                            │
#           ┌────────────────┼────────────────┐
#           ▼                                 ▼
#      BM25 Builder                    Embedding Model
#           ▼                                 ▼
#   Inverted Index                    FAISS Vector Index
#           └────────────────┬────────────────┘
#                            ▼
#                   FastAPI Search Service
#                            │
#                ┌───────────┼───────────┐
#                ▼                       ▼
#        User Authentication       ACL Filtering
#                │                       │
#                └───────────┬───────────┘
#                            ▼
#                   Hybrid Search (RRF)
#                            ▼
#                Cross-Encoder Re-ranking
#                            ▼
#                   JSON Search Results
#                            ▼
#                   Streamlit Dashboard
# ```

# ---

# # 📈 Section 12: Latency Budget

# | Component              | Latency     |
# | ---------------------- | ----------- |
# | Authentication         | 5 ms        |
# | ACL Filtering          | 2 ms        |
# | BM25 Search            | 5 ms        |
# | FAISS Search           | 10 ms       |
# | RRF Fusion             | <1 ms       |
# | Cross-Encoder (Top 20) | 120 ms      |
# | JSON Serialization     | 2 ms        |
# | **Total**              | **~145 ms** |

# Target:

# * Under 200 ms end-to-end.

# ---

# # 🧠 Section 13: Scaling Strategies

# | Problem                    | Solution                  |
# | -------------------------- | ------------------------- |
# | Millions of documents      | FAISS IVF/HNSW indexes    |
# | Slow embedding generation  | GPU embedding workers     |
# | Frequent document updates  | Micro-batch indexing      |
# | Large API traffic          | Kubernetes autoscaling    |
# | Repeated identical queries | Redis caching             |
# | Multi-company deployments  | Tenant metadata isolation |

# ---

# # 🔒 Section 14: Security Best Practices

# ## Authentication

# * Azure AD.
# * Okta.
# * OAuth2.
# * JWT tokens.

# ---

# ## Authorization

# * RBAC.
# * ACL metadata.
# * Multi-tenant filtering.

# ---

# ## Data Protection

# * HTTPS only.
# * Encrypt stored indexes.
# * Audit every search request.
# * Log unauthorized access attempts.

# ---

# # 📊 Section 15: Monitoring and Observability

# Monitor:

# | Metric          | Tool             |
# | --------------- | ---------------- |
# | API latency     | Prometheus       |
# | Error rate      | Grafana          |
# | Search traffic  | ELK Stack        |
# | Model drift     | MLflow           |
# | Index freshness | Custom dashboard |

# ---

# ## Example Dashboard

# ```text
# Enterprise Search Monitoring

# API Latency          : 145 ms
# Requests / Minute    : 520
# FAISS Index Size     : 2.4M vectors
# Cross Encoder Queue  : 14 requests
# Failed Searches      : 0.2%
# ```

# ---

# # 🧪 Section 16: Simulate End-to-End Pipeline

# ## Cell 5

# ```python
# query = (
#     "PHX-245 vector database migration"
# )

# user = "alice"

# print(
#     "User:",
#     user
# )

# print(
#     "Query:",
#     query
# )

# print(
#     "\nStep 1: Authentication ✓"
# )

# print(
#     "Step 2: ACL Filtering ✓"
# )

# print(
#     "Step 3: BM25 Retrieval ✓"
# )

# print(
#     "Step 4: FAISS Retrieval ✓"
# )

# print(
#     "Step 5: RRF Fusion ✓"
# )

# print(
#     "Step 6: Cross-Encoder Re-ranking ✓"
# )

# print(
#     "Step 7: JSON API Response ✓"
# )

# print(
#     "Step 8: Streamlit Visualization ✓"
# )
# ```

# ---

# # 🧠 Section 17: System Design Interview Cheat Sheet

# ## Q1. How would you search across Slack, Jira, and Confluence?

# **Answer:**

# > I would build a unified indexing pipeline that periodically ingests content from each source, normalizes it into a common schema, enriches it with metadata and ACL information, and indexes it into BM25 and FAISS retrieval engines.

# ---

# ## Q2. Why use Hybrid Search?

# **Answer:**

# > BM25 handles exact identifiers like project IDs and ticket numbers, while dense embeddings capture semantic similarity. Combining them using Reciprocal Rank Fusion (RRF) provides high recall and robustness.

# ---

# ## Q3. Why not use Cross-Encoder directly?

# **Answer:**

# > Cross-Encoders require one transformer pass per query-document pair, making them too expensive for large corpora. A fast retriever first narrows the candidate pool before re-ranking.

# ---

# ## Q4. How do you handle document updates?

# **Answer:**

# > I use document hashing and webhook-driven incremental indexing. Only changed documents are re-embedded and updated inside the BM25 and FAISS indexes.

# ---

# ## Q5. How do you prevent data leakage?

# **Answer:**

# > I apply ACL and tenant metadata filtering before retrieval and re-ranking so unauthorized documents never participate in BM25, FAISS, or Cross-Encoder processing.

# ---

# ## Q6. How would you scale to 50 million documents?

# **Answer:**

# > I would use distributed ingestion workers, GPU embedding services, FAISS IVF/HNSW indexes, Kubernetes autoscaling, Redis caching, and horizontally scalable API replicas behind a load balancer.

# ---

# # 📝 Section 18: Project Summary

# ## We Built

# | Notebook | Component                        |
# | -------- | -------------------------------- |
# | 01       | Enterprise Corpus Builder        |
# | 02       | ACL & Metadata Layer             |
# | 03       | BM25 Lexical Retrieval           |
# | 04       | SBERT + FAISS Semantic Retrieval |
# | 05       | Hybrid Search + RRF              |
# | 06       | Cross-Encoder Re-ranking         |
# | 07       | Secure ACL-aware Search          |
# | 08       | FastAPI Backend                  |
# | 09       | Streamlit Dashboard              |
# | 10       | Production System Design         |

# ---

# # 🏆 Final Production Architecture

# ```text
#                                   ┌───────────────────────────────┐
#                                   │   Slack / Jira / Confluence   │
#                                   │   SharePoint / Internal Wiki  │
#                                   └───────────────┬───────────────┘
#                                                   │
#                                                   ▼
#                                   ┌───────────────────────────────┐
#                                   │   ETL & Ingestion Pipeline    │
#                                   └───────────────┬───────────────┘
#                                                   │
#                                                   ▼
#                                   ┌───────────────────────────────┐
#                                   │ Metadata + ACL Enrichment     │
#                                   └───────────────┬───────────────┘
#                                                   │
#                        ┌──────────────────────────┼──────────────────────────┐
#                        ▼                                                     ▼
#             ┌───────────────────────┐                          ┌────────────────────────┐
#             │ BM25 Inverted Index   │                          │ SBERT Embedding Engine │
#             └───────────┬───────────┘                          └───────────┬────────────┘
#                         ▼                                                  ▼
#                   BM25 Search                                     FAISS Vector Index
#                         └─────────────────────┬────────────────────────────┘
#                                               ▼
#                               Reciprocal Rank Fusion (RRF)
#                                               ▼
#                                Cross-Encoder Re-ranking
#                                               ▼
#                                   FastAPI Search Backend
#                                               │
#                        ┌──────────────────────┼──────────────────────┐
#                        ▼                                             ▼
#              Authentication / SSO                          ACL / Tenant Filter
#                        └──────────────────────┬──────────────────────┘
#                                               ▼
#                                   Streamlit Dashboard
#                                               ▼
#                                    Enterprise End Users
# ```

# ---

# # 🎤 Ultimate Interview One-Liner

# > **I designed and implemented an end-to-end enterprise AI search platform that ingests documents from multiple corporate knowledge sources, enriches them with ACL-aware metadata, and builds both BM25 and SBERT+FAISS indexes. User queries are authenticated, filtered by role-based access control, processed through a Hybrid Search pipeline using Reciprocal Rank Fusion, and finally refined using a Cross-Encoder re-ranker. The solution is exposed through a FastAPI backend and visualized via a Streamlit dashboard, with support for incremental indexing, Docker deployment, Kubernetes scaling, and production-grade monitoring.**

# ---

# # 🚀 Final Learning Outcome

# After completing these 10 notebooks, you will have hands-on experience with the complete lifecycle of an **Enterprise AI Search System**:

# ```text
# Data Sources
#      │
#      ▼
# Corpus Building
#      │
#      ▼
# ACL & Metadata Enrichment
#      │
#      ▼
# BM25 Sparse Retrieval
#      │
#      ▼
# SBERT Embeddings + FAISS
#      │
#      ▼
# Hybrid Search (RRF)
#      │
#      ▼
# Cross-Encoder Re-ranking
#      │
#      ▼
# Secure ACL-aware Retrieval
#      │
#      ▼
# FastAPI Service
#      │
#      ▼
# Streamlit Dashboard
#      │
#      ▼
# Production Deployment (Docker + Kubernetes)
# ```

# ## 🧠 Resume / Interview Value

# You can confidently describe this project as:

# > **"Built a production-style Enterprise Search and Retrieval platform using Hybrid Search (BM25 + Sentence-BERT + FAISS), Reciprocal Rank Fusion (RRF), Cross-Encoder re-ranking, ACL-aware secure retrieval, FastAPI backend services, and a Streamlit frontend dashboard. Implemented metadata indexing, incremental updates, and production deployment patterns including Docker, Kubernetes, and scalable vector search architectures."**

# This is a **strong portfolio-grade GenAI + Information Retrieval + RAG systems project** that demonstrates practical understanding of retrieval systems used in modern enterprise AI applications.